# Diagnostic — does raw (non-normalized) distance find movement cosine drift hides? (colab_11 v2)

colab_11 v2's detector #1 came back INERT (held-out drift 0.0051) despite a genuinely better-fitting adapter than v1 -- a real, widening loss improvement at every checkpoint. Cosine distance is scale-invariant by construction: it measures only the angle between two vectors, never their length. If continued pretraining shifts the pooled embedding's magnitude without rotating it much, cosine distance stays near-zero while the raw distance between the same two vectors would not. This is a standalone, read-only diagnostic -- it loads the already-saved zero-shot and CPT-v2 embeddings from Drive and recomputes distance metrics on them; it re-runs no part of the training or embedding pipeline and needs no GPU.

### Setup — install `anndata`

Colab's base runtime ships `numpy`/`pandas` but not `anndata`, which is needed to read the saved `.h5ad` embeddings. `anndata` is pinned to `0.13.2` — the exact version this run used — rather than left unpinned. On a kernel where `numpy` had already been imported, an unpinned `pip install anndata` pulled a newer release that upgraded `numpy` on disk and broke the import with a load-time version skew; pinning to the version that produced these outputs keeps the install reproducible, and run on a clean base before any heavy `numpy` use it coexists with Colab's `numpy` without a skew. The resolved version prints below for the software-version record.

In [ ]:
!pip install -q "anndata==0.13.2"
from importlib.metadata import version
print("anndata", version("anndata"))

anndata 0.13.2


> **Interpretation — `anndata` installed cleanly (setup).** `anndata==0.13.2` installed on Colab's clean base and imported without the `numpy` load-time skew that an unpinned install triggered on an earlier attempt; reading the two saved `.h5ad` embeddings needs nothing more. Nothing scientific happens here — the analysis begins at 1a.

### 1a — load both saved embeddings, align by `cell_index`, reproduce the known cosine drift

Load the frozen zero-shot baseline (`glia_geneformer_zeroshot.h5ad`, colab_09) and the v2 post-CPT embedding (`glia_geneformer_cpt_aggregated_v2_seed0.h5ad`, colab_11), align them cell-for-cell by `cell_index` exactly as colab_11 §7a did, and recompute the same per-cell cosine drift metric. Reproducing 0.0050/0.0051 here before computing anything new confirms the alignment logic is correct -- if this diagnostic's own cosine number doesn't match colab_11's, nothing built on top of it can be trusted.

In [2]:
import os
import numpy as np
import pandas as pd
import anndata as ad

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
ZEROSHOT_PATH = os.path.join(DRIVE_ROOT, "geneformer", "glia_geneformer_zeroshot.h5ad")
CPT_V2_PATH   = os.path.join(DRIVE_ROOT, "geneformer", "glia_geneformer_cpt_aggregated_v2_seed0.h5ad")
for p in (ZEROSHOT_PATH, CPT_V2_PATH):
    assert os.path.exists(p), f"missing saved embedding: {p}"

zs  = ad.read_h5ad(ZEROSHOT_PATH)
cpt = ad.read_h5ad(CPT_V2_PATH)
print("zero-shot:", zs.shape, "| cpt v2:", cpt.shape)

# align by cell_index exactly as colab_11 §7a did (cpt carries the authoritative train/test split)
zs_df  = pd.DataFrame(zs.X,  index=zs.obs["cell_index"].values)
cpt_df = pd.DataFrame(cpt.X, index=cpt.obs["cell_index"].values)
common = cpt.obs["cell_index"].values
zs_aligned = zs_df.reindex(common)
assert zs_aligned.notna().all().all(), "zero-shot rows missing after cell_index alignment"

A = zs_aligned.to_numpy(dtype=np.float32)   # zero-shot
B = cpt_df.to_numpy(dtype=np.float32)       # post-CPT v2
assert A.shape == B.shape
split = cpt.obs["split"].values
test_mask = split == "test"
assert test_mask.any()

def _per_cell_cosine(X, Y):
    num = (X * Y).sum(1)
    den = np.linalg.norm(X, axis=1) * np.linalg.norm(Y, axis=1) + 1e-12
    return num / den

cos = _per_cell_cosine(A, B)
cos_drift_all  = 1.0 - float(np.median(cos))
cos_drift_test = 1.0 - float(np.median(cos[test_mask]))
print(f"reproduced cosine drift -- all: {cos_drift_all:.4f} | test: {cos_drift_test:.4f}")
print("colab_11 v2 §7a recorded  -- all: 0.0050 | test: 0.0051")


Mounted at /content/drive
zero-shot: (142588, 768) | cpt v2: (142588, 768)
reproduced cosine drift -- all: 0.0050 | test: 0.0051
colab_11 v2 §7a recorded  -- all: 0.0050 | test: 0.0051


> **Interpretation — alignment verified, cosine drift reproduced exactly (1a).** Both objects load at (142,588, 768) — the identical glia substrate and 768-dim embedding as colab_09/colab_11 — and after aligning cell-for-cell by `cell_index`, the recomputed per-cell cosine drift is 0.0050 (all) / 0.0051 (test), matching colab_11 v2 §7a's recorded 0.0050 / 0.0051 to the fourth decimal. Reproducing the original metric independently, before computing anything new, confirms the alignment logic is correct, so every downstream number here can be trusted. This cell establishes nothing about the mechanism — it is the gate that licenses 2a and 3a.

### 2a — do the two embeddings sit at different typical lengths?

Cosine distance can't see a pure magnitude shift by construction. Before testing raw distance, check the more basic question directly: does the post-CPT embedding's typical vector length (its norm) differ from the zero-shot one's? Report the median norm of each, split by held-out test vs train (the gate's actual surface) and all cells for context, plus the ratio between them -- a ratio far from 1 would mean CPT systematically stretches or shrinks the pooled vector.

In [3]:
normA = np.linalg.norm(A, axis=1)
normB = np.linalg.norm(B, axis=1)

for name, mask in [("all", np.ones(len(A), dtype=bool)), ("train", split == "train"), ("test", test_mask)]:
    mA, mB = float(np.median(normA[mask])), float(np.median(normB[mask]))
    print(f"{name:5s} | median ||zero-shot||: {mA:8.3f} | median ||cpt-v2||: {mB:8.3f} | ratio: {mB/mA:.4f}")


all   | median ||zero-shot||:    6.189 | median ||cpt-v2||:    6.225 | ratio: 1.0058
train | median ||zero-shot||:    6.194 | median ||cpt-v2||:    6.229 | ratio: 1.0057
test  | median ||zero-shot||:    6.183 | median ||cpt-v2||:    6.219 | ratio: 1.0058


> **Interpretation — no magnitude shift: CPT neither stretches nor shrinks the pooled vector (2a).** Median vector length is essentially unchanged from zero-shot to CPT-v2 — 6.189 to 6.225 (all), 6.194 to 6.229 (train), 6.183 to 6.219 (test), a ratio of 1.006 in every split. The direction of that ca. 0.6% change is consistent (CPT vectors are fractionally longer, all splits agree), but its size is negligible. This directly answers the hypothesis that motivated the diagnostic — that CPT might move the embedding's *length* while leaving its angle nearly fixed, a shift cosine distance is blind to by construction. It does not: there is no magnitude movement for cosine to have hidden. The test-split ratio (1.0058) is the gate-relevant number and it is as flat as the rest.

### 3a — raw (non-normalized) per-cell distance, on the same scale as cosine distance

Compute the raw Euclidean distance between each cell's zero-shot and post-CPT-v2 vector, then normalize it by the pair's own typical length (`||A-B|| / mean(||A||, ||B||)`) so it lands on a comparable dimensionless scale to cosine distance instead of an arbitrary raw number. If this relative distance is also ca. 0.005 on the held-out test split, magnitude movement is just as flat as direction and this diagnostic is a null -- the mechanism is more likely mean-pooling than cosine-normalization. If it's meaningfully larger than the cosine drift, a real magnitude-driven shift was there all along, one the angle-only metric structurally could not see.

In [4]:
raw_dist = np.linalg.norm(A - B, axis=1)
typical_len = (normA + normB) / 2.0
rel_dist = raw_dist / typical_len

for name, mask in [("all", np.ones(len(A), dtype=bool)), ("train", split == "train"), ("test", test_mask)]:
    print(f"{name:5s} | median raw ||A-B||: {np.median(raw_dist[mask]):8.4f} "
          f"| median relative dist: {np.median(rel_dist[mask]):.4f}")

print(f"\nfor reference -- cosine drift (test): {cos_drift_test:.4f}")


all   | median raw ||A-B||:   0.6224 | median relative dist: 0.1006
train | median raw ||A-B||:   0.6201 | median relative dist: 0.1001
test  | median raw ||A-B||:   0.6221 | median relative dist: 0.1008

for reference -- cosine drift (test): 0.0051


> **Interpretation — VERDICT: raw distance reveals nothing cosine missed; the diagnostic is a clean null (3a).** The median relative distance is 0.1006 (all) / 0.1008 (test) against a cosine drift of 0.0051 — at first glance about 20x larger, as if raw Euclidean distance had found real movement the angle-only metric could not see. It has not: the two numbers are the *same* rotation on two different scales. For vectors of near-equal length (2a: ratio 1.006) separated by a small angle theta, cosine distance = 1 - cos(theta) grows as theta-squared/2, while the relative chord `||A-B|| / mean-norm` grows as roughly theta (linearly); the algebra links them exactly, chord approximately equals sqrt(2 x cosine-distance). The identity chord approximately equals sqrt(2 x cosine-distance) is exact in the equal-length limit; with the observed 0.6% norm gap (2a) it predicts ca. 0.101, essentially identical to the observed 0.1008 — the hair's-breadth residual is medians not composing exactly (median-of-cosine, median-of-norms and median-of-relative-distance are computed independently), not a magnitude signal. Both metrics therefore agree the pooled embedding rotated by only ca. 5.8 degrees (arccos 0.9949) and barely changed length; the 0.10 figure is the trigonometric image of that same tiny rotation, not additional signal. Had a magnitude-driven shift been hidden, the observed relative distance would have *exceeded* sqrt(2 x cosine-distance), or 2a's ratio would have departed from 1 — neither occurred. **Consequence:** cosine-normalization is exonerated as the cause of the INERT verdict — it was not eating a magnitude signal, because there is no magnitude signal; the pooled vector genuinely moved very little, consistent with a structural attenuation bottleneck (as the earlier `diag_colab_11_cpt_inert` notebook argued) rather than a metric artifact. The one locus this diagnostic does not rule out is mean-pooling itself — whether per-token movement is larger before it is averaged into the single pooled vector — which would require extracting per-token hidden states from a fresh forward pass and is the only untested mechanism remaining.